In [4]:
# ------------------------------------------------------------
# To compare with CADTH, we must use THEIR prices:
#   * event, long-term and INR costs back in 2011 dollars (undo our CPI step)
#   * apixaban at its 2011 BRAND price: $1,289.44 a year (CADTH Table 1)
# Clinical inputs are already identical (we took them from CADTH).
# ------------------------------------------------------------
import pandas as pd
from markov_model import load_inputs, run_both

p, S = load_inputs()
inputs = pd.read_csv("../outputs/inputs.csv", index_col="name")
INFLATE = S["inflate_2011_to_2025"]

p2011 = dict(p)                     # copy, so the reference case stays untouched
inflated = inputs.index[inputs["source"] == "CADTH WE x CPI"]
for name in inflated:
    p2011[name] = p[name] / INFLATE  # divide by 1.3695 -> back to 2011 dollars
p2011["c_apixaban"] = 1289.44

print("Parameters deflated to 2011 dollars:", len(inflated))
print("Apixaban annual cost (2011 brand):", p2011["c_apixaban"])

Parameters deflated to 2011 dollars: 12
Apixaban annual cost (2011 brand): 1289.44


In [5]:
# ------------------------------------------------------------
# CADTH values: Guidelines for the Economic Evaluation of Health
# Technologies: Canada, 4th ed., Appendix (worked example), 2017,
# Tables 3 and 4, CHADS2 = 2 subgroup.
# ------------------------------------------------------------
apx11, war11 = run_both(p2011, S)

cadth = {
    "Warfarin cost":                  20558, "Warfarin QALYs":       7.49,
    "Warfarin life-years":             9.12, "Apixaban cost":       24665,
    "Apixaban QALYs":                  7.67, "Apixaban life-years":  9.48,
    "Incremental cost":                4107, "Incremental QALYs":   0.180,
    "ICER ($/QALY)":                  22875,
    "Strokes /1,000 (warfarin)":        242, "Strokes /1,000 (apixaban)":      201,
    "ICH /1,000 (warfarin)":             61, "ICH /1,000 (apixaban)":           31,
    "Major bleeds /1,000 (warfarin)":   298, "Major bleeds /1,000 (apixaban)": 235,
}

d_cost = apx11["cost"] - war11["cost"]
d_qaly = apx11["qaly"] - war11["qaly"]
ours = {
    "Warfarin cost": war11["cost"], "Warfarin QALYs": war11["qaly"],
    "Warfarin life-years": war11["ly"], "Apixaban cost": apx11["cost"],
    "Apixaban QALYs": apx11["qaly"], "Apixaban life-years": apx11["ly"],
    "Incremental cost": d_cost, "Incremental QALYs": d_qaly,
    "ICER ($/QALY)": d_cost / d_qaly,
    "Strokes /1,000 (warfarin)": war11["strokes"] * 1000,
    "Strokes /1,000 (apixaban)": apx11["strokes"] * 1000,
    "ICH /1,000 (warfarin)": war11["ich"] * 1000,
    "ICH /1,000 (apixaban)": apx11["ich"] * 1000,
    "Major bleeds /1,000 (warfarin)": war11["major_bleeds"] * 1000,
    "Major bleeds /1,000 (apixaban)": apx11["major_bleeds"] * 1000,
}

validation = pd.DataFrame({"CADTH (2017)": cadth, "Our model": ours})
validation["% difference"] = (validation["Our model"] / validation["CADTH (2017)"] - 1) * 100

pd.set_option("display.float_format", "{:,.2f}".format)
print(validation)
validation.to_csv("../outputs/validation_vs_cadth.csv")

                                CADTH (2017)  Our model  % difference
Warfarin cost                      20,558.00  20,875.17          1.54
Warfarin QALYs                          7.49       7.23         -3.50
Warfarin life-years                     9.12       9.13          0.10
Apixaban cost                      24,665.00  24,919.46          1.03
Apixaban QALYs                          7.67       7.46         -2.68
Apixaban life-years                     9.48       9.39         -0.97
Incremental cost                    4,107.00   4,044.28         -1.53
Incremental QALYs                       0.18       0.24         31.26
ICER ($/QALY)                      22,875.00  17,117.18        -25.17
Strokes /1,000 (warfarin)             242.00     281.75         16.43
Strokes /1,000 (apixaban)             201.00     238.61         18.71
ICH /1,000 (warfarin)                  61.00      69.80         14.43
ICH /1,000 (apixaban)                  31.00      35.52         14.58
Major bleeds /1,000 

In [6]:
# ------------------------------------------------------------
# Tests where we KNOW the right answer.
# The first two are the checks CADTH lists in its worked example (section 8).
# ------------------------------------------------------------

# CHECK 1: if every utility = 1 and no QALY losses, QALYs must equal life-years
c = dict(p)
for k in ["u_af", "u_minor_stroke", "u_rankin_3_4", "u_rankin_5"]:
    c[k] = 1.0
for k in ["dec_per_year_over_70", "dec_mi", "dec_major_bleed", "dec_minor_bleed", "dec_tia"]:
    c[k] = 0.0
a1, _ = run_both(c, S)
print(f"Check 1  QALYs = {a1['qaly']:.6f}   life-years = {a1['ly']:.6f}")
assert abs(a1["qaly"] - a1["ly"]) < 1e-9

# CHECK 2: if apixaban has NO effect (all ORs = 1) and costs the same as
# warfarin + INR, the two arms must be identical
c = dict(p)
for k in ["or_apx_stroke", "or_apx_mi", "or_apx_ich", "or_apx_mb"]:
    c[k] = 1.0
c["c_apixaban"] = p["c_warfarin"] + p["c_inr"]
a2, w2 = run_both(c, S)
print(f"Check 2  cost difference = {a2['cost'] - w2['cost']:.6f}   "
      f"QALY difference = {a2['qaly'] - w2['qaly']:.6f}")
assert abs(a2["cost"] - w2["cost"]) < 1e-6 and abs(a2["qaly"] - w2["qaly"]) < 1e-9

# CHECK 3: direction. A stronger stroke effect (lower OR) must give MORE
# QALYs and LOWER costs, and the change must be monotonic
print("\nCheck 3  OR(stroke)   incremental QALYs   incremental cost")
prev_q = None
for v in [0.5, 0.79, 1.0, 1.2]:
    c = dict(p); c["or_apx_stroke"] = v
    a3, w3 = run_both(c, S)
    dq, dc = a3["qaly"] - w3["qaly"], a3["cost"] - w3["cost"]
    print(f"           {v:<10}  {dq:>10.4f}          {dc:>10,.0f}")
    if prev_q is not None:
        assert dq < prev_q          # weaker effect -> fewer QALYs gained
    prev_q = dq

# CHECK 4: discounting. Higher rate -> fewer discounted QALYs
print("\nCheck 4  discount rate   warfarin QALYs")
for v in [0.0, 0.015, 0.03]:
    S2 = dict(S); S2["discount_rate"] = v
    _, w4 = run_both(p, S2)
    print(f"           {v:<12}  {w4['qaly']:.3f}")

print("\nAll internal validity checks passed.")

Check 1  QALYs = 9.387891   life-years = 9.387891
Check 2  cost difference = 0.000000   QALY difference = 0.000000

Check 3  OR(stroke)   incremental QALYs   incremental cost
           0.5             0.3985              -9,359
           0.79            0.2363              -5,608
           1.0             0.1251              -3,051
           1.2             0.0238                -732

Check 4  discount rate   warfarin QALYs
           0.0           7.944
           0.015         7.228
           0.03          6.619

All internal validity checks passed.
